# Trace what a proof rests on with #print axioms

*Adversarial Mathematics From First Principles*, [Part II, chapter 7](https://programmer.ie/books/math/16-chapter/) · [worked example](https://programmer.ie/examples/math/16-chapter/)

Run the cells in order. The program is a small, self-contained demonstration of the chapter's mathematics; the output saved with it is what it printed when this notebook was built.

Trust flows along the dependency graph: a result is only as well established
as everything beneath it. This core-Lean file has one unproved lemma, one
careful proof built on it, one independent result, one use of classical
logic and one false claim, and asks Lean what each rests on.

## Run it

Colab has no Lean, so the first cell downloads the official Lean 4.35.0 release (core only, about 600 MB, a minute or two). The second writes the file and the third checks it.

In [1]:
!apt-get -qq install -y zstd > /dev/null
!test -x lean-4.35.0-linux/bin/lean || curl -sSfL https://github.com/leanprover/lean4/releases/download/v4.35.0/lean-4.35.0-linux.tar.zst | tar --zstd -x
!lean-4.35.0-linux/bin/lean --version

Lean (version 4.35.0, x86_64-unknown-linux-gnu, commit eb6cafb7290e8bd5f2937522d8d804f58f02bafc, Release)


In [2]:
%%writefile proof_rests_on.lean
-- What a proof rests on: trust flows along the dependency graph (core Lean 4).

theorem stepA (n : Nat) : n + 0 = n := sorry                    -- unproved
theorem stepB (n : Nat) : (n + 0) + 0 = n := (stepA (n + 0)).trans (stepA n)
theorem stepC (n : Nat) : 0 * n = 0 := Nat.zero_mul n

#print axioms stepA   -- sorryAx
#print axioms stepB   -- sorryAx: careful proof, unproved foundation
#print axioms stepC   -- nothing: independent of stepA

-- Classical reasoning is an axiom too, and it shows up.
theorem em_used (p : Prop) : p ∨ ¬ p := Classical.em p
#print axioms em_used

-- The kernel must be seen to say no: a false statement is rejected.
theorem wrong : 2 ^ 5 + 1 = 34 := by decide

Writing proof_rests_on.lean


In [3]:
!lean-4.35.0-linux/bin/lean proof_rests_on.lean

proof_rests_on.lean:3:8: warning: declaration uses `sorry`
'stepA' depends on axioms: [sorryAx]
'stepB' depends on axioms: [sorryAx]
'stepC' does not depend on any axioms
'em_used' depends on axioms: [propext, Classical.choice, Quot.sound]
proof_rests_on.lean:16:37: error: Tactic `decide` proved that the proposition
  2 ^ 5 + 1 = 34
is false


## What the output shows

`stepB` has a correct proof of its own, but it uses `stepA`, which is `sorry`:
so it rests on `sorryAx`, and is not proved. `stepC` does not touch either and
depends on nothing. `em_used` shows that classical reasoning appears as axioms
too. The last line is the kernel seen to say no: `decide` evaluates both sides
and reports the proposition false. The same reading, at scale, is Part I,
chapter 5.

## Not shown

- `#print axioms` reports what a declaration depends on; it does not judge whether those axioms are acceptable for the claim.
- The warnings and the final error are the expected output: the file is meant to fail on its last line.

## Challenge it

Examine the dependency graph of the file. For each declaration, predict
`#print axioms` before running it, then check. Explain why `stepB` is not
proved although its own proof is correct. Then add a declaration that uses
`stepC` and `em_used` but not `stepA`, and say what it rests on.